# Sampled Scatter Decoding Orchestrator — Quality-Filtered Patient Speech

Identical to `scat_classifier_sampled_nocv.ipynb` but restricted to words that:

1. **Pass quality filtration** (`{patient}_word_df_filtered.csv`)
2. **Are spoken by the patient** (`{patient}_transcripts_annotated.csv` — LR-ASD marks segment as patient speech)

Word subset indices are saved per-patient and passed to the worker via `--word-idx-path`.

In [1]:
import json
import subprocess
from pathlib import Path

import dill as pickle
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

In [2]:
# -- Config -----------------------------------------------------------------
PATIENTS = ['YEY', 'YEZ', 'YFA', 'YFB', 'YFC', 'YFD', 'YFE', 'YFF', 'YFG', 'YFI', 'YFK', 'YFL', 'YFM', 'YFN', 'YFP', 'YFQ', 'YFR', 'YFT', 'YFU', 'YFV']

VAD_ROOT    = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new')
LEGACY_ROOT = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_out')

RUN_NAME = 'scat_xgboost_sampled_norm_filtered_patient'

CLUSTER_PREDS_OVERRIDE = {}
FRS_PATH_OVERRIDE      = {}

N_RESAMPLES  = 20
FIRST_STAGE  = [0, 1, 2]
SECOND_STAGE = list(range(3, N_RESAMPLES))
SEED_STRIDE  = 42

PYTHON_BIN  = '/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python3'
WORKER_PATH = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_decoding_analysis/scat_classifier_worker.py')

PARTITION = 'Universal'
QOS       = 'big_batch_tier'
CPUS      = 8
MEM       = '40G'
TIME      = '48:00:00'

TEST_SIZE    = 0.2
SEARCH_ITERS = 40
CV_SPLITS    = 4
N_SHUFFLES   = 50
SCORING      = 'f1_macro'
PERM_METRIC  = 'f1_macro'

print('patients:', PATIENTS)
print('worker:', WORKER_PATH)

patients: ['YEY', 'YEZ', 'YFA', 'YFB', 'YFC', 'YFD', 'YFE', 'YFF', 'YFG', 'YFI', 'YFK', 'YFL', 'YFM', 'YFN', 'YFP', 'YFQ', 'YFR', 'YFT', 'YFU', 'YFV']
worker: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_decoding_analysis/scat_classifier_worker.py


### Compute & Save Quality-Filtered Patient-Speech Word Indices Per Patient

In [3]:
def compute_filtered_patient_indices(patient: str) -> 'Path | None':
    """
    Merge quality-filtered word df with patient_speaking annotations.
    Returns path to saved .npy index file, or None if inputs are missing.
    """
    patient_root  = VAD_ROOT / patient
    filtered_csv  = patient_root / f'{patient}_word_df_filtered.csv'
    annotated_csv = patient_root / f'{patient}_transcripts_annotated.csv'
    idx_out_path  = patient_root / 'standard_decoding_analysis' / RUN_NAME / f'{patient}_filtered_patient_word_idx.npy'

    if not filtered_csv.exists():
        print(f'  [{patient}] missing {filtered_csv.name}')
        return None
    if not annotated_csv.exists():
        print(f'  [{patient}] missing {annotated_csv.name} — run add_video_diarization_df.ipynb first')
        return None

    filtered_df  = pd.read_csv(filtered_csv)
    annotated_df = pd.read_csv(annotated_csv, index_col=0)

    patient_speaking = annotated_df['patient_speaking'].astype(bool)
    filtered_df['patient_speaking'] = patient_speaking.reindex(
        filtered_df['original_word_idx']
    ).values

    patient_words = filtered_df[filtered_df['patient_speaking'] == True]
    word_idx = patient_words['original_word_idx'].values.astype(np.int64)

    idx_out_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(idx_out_path, word_idx)

    n_total   = len(filtered_df)
    n_patient = len(word_idx)
    print(f'  [{patient}] {n_patient:,} patient-speech words / {n_total:,} quality-filtered '
          f'({100*n_patient/n_total:.1f}%) → {idx_out_path.name}')
    return idx_out_path


print(f'Computing filtered patient-speech word indices for {len(PATIENTS)} patients...')
patient_idx_paths = {}
for patient in PATIENTS:
    idx_path = compute_filtered_patient_indices(patient)
    if idx_path is not None:
        patient_idx_paths[patient] = idx_path

print(f'\nReady: {len(patient_idx_paths)} / {len(PATIENTS)} patients')

Computing filtered patient-speech word indices for 20 patients...
  [YEY] missing YEY_transcripts_annotated.csv — run add_video_diarization_df.ipynb first
  [YEZ] missing YEZ_transcripts_annotated.csv — run add_video_diarization_df.ipynb first
  [YFA] missing YFA_transcripts_annotated.csv — run add_video_diarization_df.ipynb first
  [YFB] missing YFB_transcripts_annotated.csv — run add_video_diarization_df.ipynb first
  [YFC] 17,828 patient-speech words / 292,428 quality-filtered (6.1%) → YFC_filtered_patient_word_idx.npy
  [YFD] 16,307 patient-speech words / 388,734 quality-filtered (4.2%) → YFD_filtered_patient_word_idx.npy
  [YFE] 3,845 patient-speech words / 199,717 quality-filtered (1.9%) → YFE_filtered_patient_word_idx.npy
  [YFF] 11,244 patient-speech words / 228,795 quality-filtered (4.9%) → YFF_filtered_patient_word_idx.npy
  [YFG] 29 patient-speech words / 48,567 quality-filtered (0.1%) → YFG_filtered_patient_word_idx.npy
  [YFI] 5,830 patient-speech words / 126,297 quality-f

In [8]:
# -- Path resolution helpers -------------------------------------------------
def first_existing(paths):
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None


def patient_root(patient: str) -> Path:
    return VAD_ROOT / patient


def output_root(patient: str) -> Path:
    out = patient_root(patient) / 'standard_decoding_analysis' / RUN_NAME
    out.mkdir(parents=True, exist_ok=True)
    return out


def logs_dir(patient: str) -> Path:
    p = patient_root(patient) / 'decoding_slurm_logs' / RUN_NAME
    p.mkdir(parents=True, exist_ok=True)
    return p


def scripts_dir(patient: str) -> Path:
    p = patient_root(patient) / 'decoding_slurm_scripts' / RUN_NAME
    p.mkdir(parents=True, exist_ok=True)
    return p


def resolve_patient_inputs(patient: str) -> dict:
    root = patient_root(patient)
    cluster_override = CLUSTER_PREDS_OVERRIDE.get(patient)
    frs_override     = FRS_PATH_OVERRIDE.get(patient)

    new_clusters = first_existing([root / 'embeddings' / 'semantic_cluster_predictions.npy'])
    new_frs = first_existing([
        root / 'neural_embeddings' / 'word_frs.npy',
        root / 'all_convo_recording' / 'word_avg_frs.npy',
    ])
    legacy_frs = first_existing([LEGACY_ROOT / patient / 'all_convo_recording' / 'word_avg_frs.npy'])

    if cluster_override is not None or frs_override is not None:
        cluster_preds_path = first_existing([cluster_override, new_clusters])
        frs_path           = first_existing([frs_override, new_frs, legacy_frs])
        input_source = 'override'
    elif new_clusters is not None and new_frs is not None:
        cluster_preds_path = new_clusters
        frs_path           = new_frs
        input_source = 'vad_new'
    else:
        cluster_preds_path = None
        frs_path           = None
        input_source = 'missing'

    word_idx_path = patient_idx_paths.get(patient)
    out = output_root(patient)
    return {
        'patient': patient,
        'patient_root': root,
        'input_source': input_source,
        'cluster_preds_path': cluster_preds_path,
        'frs_path': frs_path,
        'word_idx_path': word_idx_path,
        'output_root': out,
        'consensus_params_json': out / 'consensus_best_params.json',
        'aggregate_csv': out / 'scat_sampled_all_resamples.csv',
        'aggregate_pkl': out / 'scat_sampled_all_resamples.pkl',
        'logs_dir': logs_dir(patient),
        'scripts_dir': scripts_dir(patient),
    }


def choose_consensus_params(param_options: list) -> dict:
    params = dict(param_options[0])
    params['max_depth']        = int(np.min([p['max_depth'] for p in param_options]))
    params['min_child_weight'] = int(np.max([p['min_child_weight'] for p in param_options]))
    params['gamma']            = float(np.max([p['gamma'] for p in param_options]))
    params['reg_lambda']       = float(np.max([p['reg_lambda'] for p in param_options]))
    params['reg_alpha']        = float(np.max([p['reg_alpha'] for p in param_options]))
    params['subsample']        = float(np.median([p['subsample'] for p in param_options]))
    params['colsample_bytree'] = float(np.median([p['colsample_bytree'] for p in param_options]))
    params['learning_rate']    = float(np.median([p['learning_rate'] for p in param_options]))
    params['n_estimators']     = int(np.median([p['n_estimators'] for p in param_options]))
    params['tree_method'] = 'hist'
    params['device'] = 'cpu'
    params.pop('predictor', None)
    params.pop('use_label_encoder', None)
    return params

In [9]:
# -- Status helpers ----------------------------------------------------------
def resample_paths(out_root: Path, r: int) -> dict:
    return {
        'pkl':              out_root / f'scat_sampled_resample_{r}_.pkl',
        'summary_json':     out_root / f'summary_resample_{r}.json',
        'best_params_json': out_root / f'best_params_resample_{r}.json',
        'success':          out_root / f'resample_{r}_SUCCESS',
        'error':            out_root / f'resample_{r}_error.txt',
    }


def build_patient_status(patient: str) -> pd.DataFrame:
    info = resolve_patient_inputs(patient)
    rows = []
    for r in range(N_RESAMPLES):
        paths = resample_paths(info['output_root'], r)
        rows.append({
            'patient': patient,
            'resample_idx': r,
            'stage': 'hyperparam' if r in FIRST_STAGE else 'fixed',
            'seed': r * SEED_STRIDE,
            'cluster_preds_path':    info['cluster_preds_path'],
            'frs_path':              info['frs_path'],
            'word_idx_path':         info['word_idx_path'],
            'has_inputs':            info['cluster_preds_path'] is not None and info['frs_path'] is not None and info['word_idx_path'] is not None,
            'consensus_params_json': info['consensus_params_json'],
            'consensus_ready':       info['consensus_params_json'].exists(),
            'pkl_path':              paths['pkl'],
            'summary_json':          paths['summary_json'],
            'best_params_json':      paths['best_params_json'],
            'success_path':          paths['success'],
            'error_path':            paths['error'],
            'has_pkl':               paths['pkl'].exists(),
            'has_summary':           paths['summary_json'].exists(),
            'has_best_params':       paths['best_params_json'].exists(),
            'has_success':           paths['success'].exists(),
            'has_error':             paths['error'].exists(),
        })
    df = pd.DataFrame(rows)
    first_three_done = bool(df[df['resample_idx'].isin(FIRST_STAGE)]['has_success'].all()) if not df.empty else False
    df['first_three_done'] = first_three_done
    df['ready_phase1'] = df['has_inputs'] & df['resample_idx'].isin(FIRST_STAGE) & ~df['has_success']
    df['ready_phase2'] = df['has_inputs'] & df['resample_idx'].isin(SECOND_STAGE) & first_three_done & df['consensus_ready'] & ~df['has_success']
    return df


status_df = pd.concat([build_patient_status(pt) for pt in PATIENTS], ignore_index=True)
status_df[['patient','resample_idx','stage','has_inputs','has_success','has_error','has_best_params','consensus_ready','ready_phase1','ready_phase2']]

,patient,resample_idx,stage,has_inputs,has_success,has_error,has_best_params,consensus_ready,ready_phase1,ready_phase2
0,YEY,0,hyperparam,False,False,False,False,False,False,False
1,YEY,1,hyperparam,False,False,False,False,False,False,False
2,YEY,2,hyperparam,False,False,False,False,False,False,False
3,YEY,3,fixed,False,False,False,False,False,False,False
4,YEY,4,fixed,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
395,YFV,15,fixed,True,True,False,True,True,False,False
396,YFV,16,fixed,True,True,False,True,True,False,False
397,YFV,17,fixed,True,True,False,True,True,False,False
398,YFV,18,fixed,True,True,False,True,True,False,False


In [10]:
# -- Patient input summary ---------------------------------------------------
patient_info_df = pd.DataFrame([resolve_patient_inputs(pt) for pt in PATIENTS])
patient_info_df[['patient', 'input_source', 'cluster_preds_path', 'frs_path', 'word_idx_path', 'output_root']]

,patient,input_source,cluster_preds_path,frs_path,word_idx_path,output_root
0,YEY,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,None,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
1,YEZ,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,None,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
2,YFA,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,None,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
3,YFB,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,None,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
4,YFC,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
5,YFD,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
6,YFE,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
7,YFF,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
8,YFG,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
9,YFI,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...


In [11]:
# -- Submit first 3 resamples per patient -----------------------------------
DRY_RUN    = False
SUBMIT_ONE = False

submitted = []
failed    = []

for patient in PATIENTS:
    info = resolve_patient_inputs(patient)
    patient_status = build_patient_status(patient)
    for _, row in patient_status[patient_status['ready_phase1']].iterrows():
        r = int(row['resample_idx'])
        word_idx_arg = f'--word-idx-path "{info["word_idx_path"]}"' if info['word_idx_path'] else ''
        sbatch_text = f'''#!/bin/bash
#SBATCH --job-name=scat_fp_{patient}_{r}
#SBATCH --partition={PARTITION}
#SBATCH --qos={QOS}
#SBATCH --cpus-per-task={CPUS}
#SBATCH --mem={MEM}
#SBATCH --time={TIME}
#SBATCH --output={info['logs_dir']}/{patient}_r{r}_%j.out
#SBATCH --error={info['logs_dir']}/{patient}_r{r}_%j.err

set -eo pipefail

echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "PATIENT: {patient}"
echo "RESAMPLE: {r}"

{PYTHON_BIN} {WORKER_PATH}   --patient {patient}   --resample-idx {r}   --cluster-preds-path "{info['cluster_preds_path']}"   --frs-path "{info['frs_path']}"   --out-dir "{info['output_root']}"   {word_idx_arg}   --test-size {TEST_SIZE}   --n-iter {SEARCH_ITERS}   --cv-splits {CV_SPLITS}   --n-shuffles {N_SHUFFLES}   --seed-stride {SEED_STRIDE}   --n-jobs {CPUS * 2}   --scoring {SCORING}   --perm-metric {PERM_METRIC}

echo "END: $(date)"
'''
        sbatch_path = info['scripts_dir'] / f'{patient}_resample_{r}.sbatch'
        sbatch_path.write_text(sbatch_text)
        if DRY_RUN:
            submitted.append((patient, r, 'dry-run'))
            print(f'dry-run: {patient} r={r}')
        else:
            try:
                res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
                submitted.append((patient, r, res.stdout.strip()))
                print(f'submitted: {patient} r={r} -> {res.stdout.strip()}')
            except subprocess.CalledProcessError as exc:
                failed.append((patient, r, exc.stderr.strip()))
                print(f'FAILED SUBMISSION: {patient} r={r}\n{exc.stderr}')
        if SUBMIT_ONE and submitted:
            break
    if SUBMIT_ONE and submitted:
        break

print(f'submitted={len(submitted)}, failed={len(failed)}')

submitted: YFG r=0 -> Submitted batch job 290876
submitted: YFG r=1 -> Submitted batch job 290877
submitted: YFG r=2 -> Submitted batch job 290878
submitted: YFN r=0 -> Submitted batch job 290879
submitted: YFN r=1 -> Submitted batch job 290880
submitted: YFN r=2 -> Submitted batch job 290881
submitted=6, failed=0


In [12]:
# -- Build consensus params once resamples 0..2 finish ----------------------
built   = []
blocked = []
for patient in PATIENTS:
    info = resolve_patient_inputs(patient)
    patient_status = build_patient_status(patient)
    if info['consensus_params_json'].exists():
        print(f'consensus already exists for {patient}: {info["consensus_params_json"]}')
        continue
    first_three = patient_status[patient_status['resample_idx'].isin(FIRST_STAGE)]
    if not bool(first_three['has_success'].all()):
        blocked.append(patient)
        print(f'waiting on first three for {patient}')
        continue
    param_options = []
    for r in FIRST_STAGE:
        path = info['output_root'] / f'best_params_resample_{r}.json'
        param_options.append(json.loads(path.read_text()))
    consensus = choose_consensus_params(param_options)
    info['consensus_params_json'].write_text(json.dumps(consensus, indent=2))
    built.append(patient)
    print(f'built consensus params for {patient} -> {info["consensus_params_json"]}')

print('built:', built)
print('blocked:', blocked)

waiting on first three for YEY
waiting on first three for YEZ
waiting on first three for YFA
waiting on first three for YFB
consensus already exists for YFC: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFC/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_patient/consensus_best_params.json
consensus already exists for YFD: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFD/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_patient/consensus_best_params.json
consensus already exists for YFE: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFE/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_patient/consensus_best_params.json
consensus already exists for YFF: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFF/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_patient/consensus_best_params.json
waiting on first three for YFG
consensus already exists for YFI: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFI/standar

In [13]:
# -- Submit remaining 17 resamples after consensus exists -------------------
DRY_RUN    = False
SUBMIT_ONE = False

submitted = []
failed    = []

for patient in PATIENTS:
    info = resolve_patient_inputs(patient)
    patient_status = build_patient_status(patient)
    for _, row in patient_status[patient_status['ready_phase2']].iterrows():
        r = int(row['resample_idx'])
        word_idx_arg = f'--word-idx-path "{info["word_idx_path"]}"' if info['word_idx_path'] else ''
        sbatch_text = f'''#!/bin/bash
#SBATCH --job-name=scat_fp_{patient}_{r}
#SBATCH --partition={PARTITION}
#SBATCH --qos={QOS}
#SBATCH --cpus-per-task={CPUS}
#SBATCH --mem={MEM}
#SBATCH --time={TIME}
#SBATCH --output={info['logs_dir']}/{patient}_r{r}_%j.out
#SBATCH --error={info['logs_dir']}/{patient}_r{r}_%j.err

set -eo pipefail

echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "PATIENT: {patient}"
echo "RESAMPLE: {r}"

{PYTHON_BIN} {WORKER_PATH}   --patient {patient}   --resample-idx {r}   --cluster-preds-path "{info['cluster_preds_path']}"   --frs-path "{info['frs_path']}"   --out-dir "{info['output_root']}"   --params-json "{info['consensus_params_json']}"   {word_idx_arg}   --test-size {TEST_SIZE}   --n-iter {SEARCH_ITERS}   --cv-splits {CV_SPLITS}   --n-shuffles {N_SHUFFLES}   --seed-stride {SEED_STRIDE}   --n-jobs {CPUS * 2}   --scoring {SCORING}   --perm-metric {PERM_METRIC}

echo "END: $(date)"
'''
        sbatch_path = info['scripts_dir'] / f'{patient}_resample_{r}.sbatch'
        sbatch_path.write_text(sbatch_text)
        if DRY_RUN:
            submitted.append((patient, r, 'dry-run'))
            print(f'dry-run: {patient} r={r}')
        else:
            try:
                res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
                submitted.append((patient, r, res.stdout.strip()))
                print(f'submitted: {patient} r={r} -> {res.stdout.strip()}')
            except subprocess.CalledProcessError as exc:
                failed.append((patient, r, exc.stderr.strip()))
                print(f'FAILED SUBMISSION: {patient} r={r}\n{exc.stderr}')
        if SUBMIT_ONE and submitted:
            break
    if SUBMIT_ONE and submitted:
        break

print(f'submitted={len(submitted)}, failed={len(failed)}')

submitted=0, failed=0


In [14]:
# -- Error inspection --------------------------------------------------------
for patient in PATIENTS:
    patient_status = build_patient_status(patient)
    errs = patient_status[patient_status['has_error']]
    if errs.empty:
        print(f'no error files for {patient}')
        continue
    print(f'errors for {patient}:')
    display(errs[['resample_idx', 'error_path']])

no error files for YEY
no error files for YEZ
no error files for YFA
no error files for YFB
no error files for YFC
no error files for YFD
no error files for YFE
no error files for YFF
errors for YFG:


,resample_idx,error_path
0,0,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
1,1,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
2,2,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...


no error files for YFI
no error files for YFK
no error files for YFL
no error files for YFM
errors for YFN:


,resample_idx,error_path
0,0,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
1,1,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
2,2,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...


no error files for YFP
no error files for YFQ
no error files for YFR
no error files for YFT
no error files for YFU
no error files for YFV


In [15]:
# -- Aggregate full patient results once all 20 finish ----------------------
for patient in PATIENTS:
    info = resolve_patient_inputs(patient)
    patient_status = build_patient_status(patient)
    if not bool(patient_status['has_success'].all()):
        print(f'skipping aggregate for {patient}: not all resamples finished')
        continue

    summary_rows = []
    aggregate    = {}
    for r in range(N_RESAMPLES):
        with open(info['output_root'] / f'summary_resample_{r}.json') as f:
            summary_rows.append(json.load(f))
        with open(info['output_root'] / f'scat_sampled_resample_{r}_.pkl', 'rb') as f:
            aggregate[f'resample_{r}'] = pickle.load(f)

    summary_df = pd.DataFrame(summary_rows).sort_values('resample_idx').reset_index(drop=True)
    summary_df.to_csv(info['aggregate_csv'], index=False)
    with open(info['aggregate_pkl'], 'wb') as f:
        pickle.dump(aggregate, f)

    print(f'aggregated {patient}')
    print(f"  csv: {info['aggregate_csv']}")
    print(f"  pkl: {info['aggregate_pkl']}")
    display(summary_df)

skipping aggregate for YEY: not all resamples finished
skipping aggregate for YEZ: not all resamples finished
skipping aggregate for YFA: not all resamples finished
skipping aggregate for YFB: not all resamples finished
aggregated YFC
  csv: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFC/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_patient/scat_sampled_all_resamples.csv
  pkl: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFC/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_patient/scat_sampled_all_resamples.pkl


,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YFC,0,0,4865,4764,953,0.229801,0.187496,0.180834,0.229801,0.619997,0.113290,0.129066,2.312831e-24,f1_macro,0.180834,0.019608,0.171362,shuffle_training_X_only
1,YFC,1,42,4865,4760,952,0.210084,0.180212,0.176199,0.210084,0.620348,0.113258,0.129202,7.572200e-18,f1_macro,0.176199,0.019608,0.168838,shuffle_training_X_only
2,YFC,2,84,4865,4763,953,0.225603,0.181677,0.169612,0.225603,0.624003,0.113279,0.129066,6.640438e-23,f1_macro,0.169612,0.019608,0.167084,shuffle_training_X_only
3,YFC,3,126,4865,4762,953,0.214061,0.175094,0.168841,0.214061,0.613564,0.113279,0.129066,4.288674e-19,f1_macro,0.168841,0.019608,NaN,shuffle_training_X_only
4,YFC,4,168,4865,4769,954,0.240042,0.196821,0.187574,0.240042,0.609123,0.113311,0.128931,4.049575e-28,f1_macro,0.187574,0.019608,NaN,shuffle_training_X_only
5,YFC,5,210,4865,4768,954,0.216981,0.179353,0.169265,0.216981,0.607189,0.113313,0.129979,4.942198e-20,f1_macro,0.169265,0.019608,NaN,shuffle_training_X_only
6,YFC,6,252,4865,4767,954,0.224319,0.184846,0.175728,0.224319,0.627282,0.113311,0.128931,1.798293e-22,f1_macro,0.175728,0.019608,NaN,shuffle_training_X_only
7,YFC,7,294,4865,4754,951,0.214511,0.177928,0.170096,0.214511,0.609617,0.113221,0.129338,3.176550e-19,f1_macro,0.170096,0.019608,NaN,shuffle_training_X_only
8,YFC,8,336,4865,4763,953,0.196222,0.160237,0.153509,0.196222,0.601171,0.113288,0.129066,7.655119e-14,f1_macro,0.153509,0.019608,NaN,shuffle_training_X_only
9,YFC,9,378,4865,4765,953,0.210913,0.171919,0.162632,0.210913,0.597393,0.113101,0.129066,3.493996e-18,f1_macro,0.162632,0.019608,NaN,shuffle_training_X_only


aggregated YFD
  csv: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFD/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_patient/scat_sampled_all_resamples.csv
  pkl: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFD/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_patient/scat_sampled_all_resamples.pkl


,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YFD,0,0,3504,3504,701,0.165478,0.133122,0.137540,0.165478,0.557020,0.12362,0.1398,0.000724,f1_macro,0.137540,0.019608,0.127295,shuffle_training_X_only
1,YFD,1,42,3504,3504,701,0.164051,0.126586,0.124134,0.164051,0.562480,0.12362,0.1398,0.001038,f1_macro,0.124134,0.039216,0.125684,shuffle_training_X_only
2,YFD,2,84,3504,3504,701,0.158345,0.121283,0.118903,0.158345,0.554170,0.12362,0.1398,0.003988,f1_macro,0.118903,0.039216,0.129684,shuffle_training_X_only
3,YFD,3,126,3504,3504,701,0.158345,0.122068,0.120785,0.158345,0.569240,0.12362,0.1398,0.003988,f1_macro,0.120785,0.019608,NaN,shuffle_training_X_only
4,YFD,4,168,3504,3504,701,0.164051,0.122525,0.116010,0.164051,0.578505,0.12362,0.1398,0.001038,f1_macro,0.116010,0.019608,NaN,shuffle_training_X_only
5,YFD,5,210,3504,3504,701,0.162625,0.119998,0.113608,0.162625,0.550023,0.12362,0.1398,0.001474,f1_macro,0.113608,0.019608,NaN,shuffle_training_X_only
6,YFD,6,252,3504,3504,701,0.155492,0.114693,0.106281,0.155492,0.570945,0.12362,0.1398,0.007375,f1_macro,0.106281,0.078431,NaN,shuffle_training_X_only
7,YFD,7,294,3504,3504,701,0.175464,0.130516,0.124670,0.175464,0.595439,0.12362,0.1398,0.000044,f1_macro,0.124670,0.019608,NaN,shuffle_training_X_only
8,YFD,8,336,3504,3504,701,0.184023,0.134394,0.125362,0.184023,0.566027,0.12362,0.1398,0.000003,f1_macro,0.125362,0.019608,NaN,shuffle_training_X_only
9,YFD,9,378,3504,3504,701,0.152639,0.112541,0.105170,0.152639,0.555881,0.12362,0.1398,0.013119,f1_macro,0.105170,0.117647,NaN,shuffle_training_X_only


aggregated YFE
  csv: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFE/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_patient/scat_sampled_all_resamples.csv
  pkl: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFE/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_patient/scat_sampled_all_resamples.pkl


...

,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YFT,0,0,6125,5181,1037,0.199614,0.155204,0.147652,0.199614,0.586569,0.118589,0.141755,5.878789e-14,f1_macro,0.147652,0.019608,0.137605,shuffle_training_X_only
1,YFT,1,42,6125,5171,1035,0.197101,0.154680,0.148370,0.197101,0.576821,0.118527,0.142029,2.949027e-13,f1_macro,0.148370,0.019608,0.142888,shuffle_training_X_only
2,YFT,2,84,6125,5176,1036,0.175676,0.136379,0.130347,0.175676,0.588062,0.118702,0.141892,5.444562e-08,f1_macro,0.130347,0.019608,0.143547,shuffle_training_X_only
3,YFT,3,126,6125,5173,1035,0.164251,0.129262,0.126806,0.164251,0.599555,0.118665,0.142029,9.084838e-06,f1_macro,0.126806,0.019608,NaN,shuffle_training_X_only
4,YFT,4,168,6125,5145,1029,0.193392,0.155131,0.153792,0.193392,0.593866,0.118487,0.142857,3.292145e-12,f1_macro,0.153792,0.019608,NaN,shuffle_training_X_only
5,YFT,5,210,6125,5196,1040,0.181731,0.141063,0.136256,0.181731,0.584106,0.118730,0.142308,2.369207e-09,f1_macro,0.136256,0.019608,NaN,shuffle_training_X_only
6,YFT,6,252,6125,5163,1033,0.185866,0.145632,0.140194,0.185866,0.584793,0.118602,0.142304,2.603974e-10,f1_macro,0.140194,0.019608,NaN,shuffle_training_X_only
7,YFT,7,294,6125,5181,1037,0.176471,0.139176,0.135758,0.176471,0.586096,0.118604,0.141755,3.427984e-08,f1_macro,0.135758,0.019608,NaN,shuffle_training_X_only
8,YFT,8,336,6125,5171,1035,0.189372,0.157413,0.160502,0.189372,0.584075,0.118527,0.142029,3.221965e-11,f1_macro,0.160502,0.019608,NaN,shuffle_training_X_only
9,YFT,9,378,6125,5169,1034,0.170213,0.130660,0.124241,0.170213,0.568547,0.118611,0.141199,6.912999e-07,f1_macro,0.124241,0.019608,NaN,shuffle_training_X_only


aggregated YFU
  csv: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFU/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_patient/scat_sampled_all_resamples.csv
  pkl: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFU/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_patient/scat_sampled_all_resamples.pkl


,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YFU,0,0,682,682,137,0.160584,0.128158,0.124859,0.160584,0.515157,0.115669,0.131387,0.070190,f1_macro,0.124859,0.098039,0.145587,shuffle_training_X_only
1,YFU,1,42,682,682,137,0.167883,0.145079,0.146694,0.167883,0.552656,0.115669,0.131387,0.042983,f1_macro,0.146694,0.039216,0.140465,shuffle_training_X_only
2,YFU,2,84,682,682,137,0.145985,0.118816,0.109417,0.145985,0.517121,0.115669,0.131387,0.163772,f1_macro,0.109417,0.156863,0.153116,shuffle_training_X_only
3,YFU,3,126,682,682,137,0.197080,0.166280,0.166773,0.197080,0.614259,0.115669,0.131387,0.003917,f1_macro,0.166773,0.019608,NaN,shuffle_training_X_only
4,YFU,4,168,682,682,137,0.175182,0.137704,0.124037,0.175182,0.605293,0.115669,0.131387,0.025190,f1_macro,0.124037,0.117647,NaN,shuffle_training_X_only
5,YFU,5,210,682,682,137,0.189781,0.157747,0.148843,0.189781,0.553230,0.115669,0.131387,0.007600,f1_macro,0.148843,0.058824,NaN,shuffle_training_X_only
6,YFU,6,252,682,682,137,0.211679,0.184075,0.181742,0.211679,0.599568,0.115669,0.131387,0.000920,f1_macro,0.181742,0.019608,NaN,shuffle_training_X_only
7,YFU,7,294,682,682,137,0.124088,0.104623,0.098010,0.124088,0.544834,0.115669,0.131387,0.417743,f1_macro,0.098010,0.313725,NaN,shuffle_training_X_only
8,YFU,8,336,682,682,137,0.131387,0.108082,0.102639,0.131387,0.581504,0.115669,0.131387,0.319546,f1_macro,0.102639,0.352941,NaN,shuffle_training_X_only
9,YFU,9,378,682,682,137,0.189781,0.155273,0.143729,0.189781,0.556971,0.115669,0.131387,0.007600,f1_macro,0.143729,0.058824,NaN,shuffle_training_X_only


aggregated YFV
  csv: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFV/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_patient/scat_sampled_all_resamples.csv
  pkl: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFV/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered_patient/scat_sampled_all_resamples.pkl


,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YFV,0,0,2120,1704,341,0.155425,0.131695,0.131431,0.155425,0.569089,0.112856,0.143695,0.010466,f1_macro,0.131431,0.019608,0.137523,shuffle_training_X_only
1,YFV,1,42,2120,1711,343,0.151603,0.125330,0.123872,0.151603,0.544868,0.113312,0.145773,0.018609,f1_macro,0.123872,0.039216,0.135950,shuffle_training_X_only
2,YFV,2,84,2120,1717,344,0.174419,0.140890,0.130762,0.174419,0.563218,0.113575,0.145349,0.000529,f1_macro,0.130762,0.019608,0.145580,shuffle_training_X_only
3,YFV,3,126,2120,1689,338,0.180473,0.148581,0.143738,0.180473,0.579827,0.112689,0.142012,0.000153,f1_macro,0.143738,0.019608,NaN,shuffle_training_X_only
4,YFV,4,168,2120,1713,343,0.186589,0.152988,0.147420,0.186589,0.554931,0.113159,0.142857,0.000044,f1_macro,0.147420,0.019608,NaN,shuffle_training_X_only
5,YFV,5,210,2120,1713,343,0.160350,0.138009,0.139065,0.160350,0.593824,0.113142,0.145773,0.005222,f1_macro,0.139065,0.019608,NaN,shuffle_training_X_only
6,YFV,6,252,2120,1711,343,0.183673,0.157041,0.153648,0.183673,0.570478,0.113040,0.142857,0.000079,f1_macro,0.153648,0.019608,NaN,shuffle_training_X_only
7,YFV,7,294,2120,1714,343,0.180758,0.146870,0.140199,0.180758,0.564582,0.113312,0.145773,0.000152,f1_macro,0.140199,0.039216,NaN,shuffle_training_X_only
8,YFV,8,336,2120,1711,343,0.186589,0.142388,0.130337,0.186589,0.562779,0.112989,0.139942,0.000043,f1_macro,0.130337,0.019608,NaN,shuffle_training_X_only
9,YFV,9,378,2120,1718,344,0.191860,0.154945,0.144503,0.191860,0.570671,0.113288,0.145349,0.000014,f1_macro,0.144503,0.019608,NaN,shuffle_training_X_only
